# 05 – Case table and late-order prediction

Goal: predict at order creation whether a purchase order will **not be cleared within 126 days**.
Run the cells from top to bottom.

In [2]:
import duckdb
con = duckdb.connect()
con.execute("""
CREATE OR REPLACE VIEW events_ord AS
SELECT *, file_row_number AS rn
FROM read_parquet('../data/bpi2019.parquet', file_row_number=true)
WHERE "case:concept:name" NOT IN (
  SELECT DISTINCT "case:concept:name"
  FROM read_parquet('../data/bpi2019.parquet')
  WHERE "time:timestamp" < TIMESTAMPTZ '2017-01-01')
""")

## 1. Case table (one row per order)

In [4]:
cols = con.sql("DESCRIBE SELECT * FROM events_ord").df()
cols[cols.column_name.str.startswith("case:")][["column_name", "column_type"]]

,column_name,column_type
5,case:Spend area text,VARCHAR
6,case:Company,VARCHAR
7,case:Document Type,VARCHAR
8,case:Sub spend area text,VARCHAR
9,case:Purchasing Document,VARCHAR
10,case:Purch. Doc. Category name,VARCHAR
11,case:Vendor,VARCHAR
12,case:Item Type,VARCHAR
13,case:Item Category,VARCHAR
14,case:Spend classification text,VARCHAR


In [5]:
con.execute("""
CREATE OR REPLACE TABLE cases AS
SELECT
  "case:concept:name"                         AS case_id,
  MIN("time:timestamp")                       AS start_ts,
  date_diff('day', MIN("time:timestamp"), MAX("time:timestamp")) AS duration_days,
  ANY_VALUE("case:Company")                   AS company,
  ANY_VALUE("case:Document Type")             AS doc_type,
  ANY_VALUE("case:Item Category")             AS item_category,
  ANY_VALUE("case:Item Type")                 AS item_type,
  ANY_VALUE("case:Spend area text")           AS spend_area,
  ANY_VALUE("case:GR-Based Inv. Verif.")      AS gr_based,
  ANY_VALUE("case:Source")                    AS source
FROM events_ord
GROUP BY 1
""")
con.sql("SELECT COUNT(*) AS cases FROM cases").df()

,cases
0,251654


## 2. How long do orders take, and where does the log stop?
Orders created after ~Aug 2018 are cut off by the end of the log (right-censoring), so they must not be used.

In [6]:
con.sql("""
SELECT
  MEDIAN(duration_days)                AS median,
  quantile_cont(duration_days, 0.75)   AS p75,
  quantile_cont(duration_days, 0.90)   AS p90,
  MAX(duration_days)                   AS maximum
FROM cases
""").df()

,median,p75,p90,maximum
0,64.0,98.0,126.0,819


In [7]:
con.sql("""
WITH done AS (
  SELECT "case:concept:name" AS case_id,
         COUNT(*) FILTER (WHERE "concept:name" = 'Clear Invoice') > 0 AS has_clear
  FROM events_ord GROUP BY 1)
SELECT date_trunc('month', c.start_ts) AS start_month,
       COUNT(*)                                   AS cases,
       ROUND(100.0 * AVG(d.has_clear::INT), 1)    AS pct_with_clear_invoice,
       MEDIAN(c.duration_days)                    AS median_days
FROM cases c JOIN done d USING (case_id)
GROUP BY 1 ORDER BY 1
""").df()

,start_month,cases,pct_with_clear_invoice,median_days
0,2017-01-01 00:00:00+01:00,13,100.0,371.0
1,2017-02-01 00:00:00+01:00,3,33.3,401.0
2,2017-03-01 00:00:00+01:00,2,50.0,432.0
3,2017-04-01 00:00:00+02:00,8,100.0,386.0
4,2017-05-01 00:00:00+02:00,3,100.0,437.0
5,2017-06-01 00:00:00+02:00,2,100.0,299.5
6,2017-07-01 00:00:00+02:00,1,100.0,486.0
7,2017-08-01 00:00:00+02:00,4,100.0,394.0
8,2017-09-01 00:00:00+02:00,5,100.0,356.0
9,2017-10-01 00:00:00+02:00,5,100.0,448.0


In [8]:
con.sql("""SELECT MIN("time:timestamp") AS first_ts, MAX("time:timestamp") AS last_ts
           FROM events_ord""").df()

,first_ts,last_ts
0,2017-01-05 23:59:00+01:00,2020-04-09 23:59:00+02:00


In [9]:
con.sql("""
WITH clr AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
  FROM events_ord GROUP BY 1)
SELECT date_trunc('month', c.start_ts) AS start_month,
       COUNT(*) AS cases,
       ROUND(100.0 * AVG((clr.clear_ts IS NULL OR date_diff('day', c.start_ts, clr.clear_ts) > 90)::INT), 1)  AS pct_late_90,
       ROUND(100.0 * AVG((clr.clear_ts IS NULL OR date_diff('day', c.start_ts, clr.clear_ts) > 126)::INT), 1) AS pct_late_126
FROM cases c JOIN clr USING (case_id)
WHERE c.start_ts >= TIMESTAMPTZ '2018-01-01'
GROUP BY 1 ORDER BY 1
""").df()

,start_month,cases,pct_late_90,pct_late_126
0,2018-01-01 00:00:00+01:00,21243,48.2,24.8
1,2018-02-01 00:00:00+01:00,20915,44.4,21.2
2,2018-03-01 00:00:00+01:00,24850,52.0,25.4
3,2018-04-01 00:00:00+02:00,22460,48.2,25.7
4,2018-05-01 00:00:00+02:00,22966,48.2,22.1
5,2018-06-01 00:00:00+02:00,22305,49.7,19.6
6,2018-07-01 00:00:00+02:00,21081,48.6,22.8
7,2018-08-01 00:00:00+02:00,20187,50.7,25.7
8,2018-09-01 00:00:00+02:00,18227,51.2,29.5
9,2018-10-01 00:00:00+02:00,21834,55.4,49.1


## 3. First cohort and target
Target `is_late` = not cleared within 126 days (p90 of duration), or never cleared. Time-based split: train Jan–Jun 2018, test from Jul 2018.

In [10]:
con.execute("""
CREATE OR REPLACE TABLE ml_cases AS
WITH clr AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
  FROM events_ord GROUP BY 1)
SELECT c.case_id, c.start_ts,
       c.company, c.doc_type, c.item_category, c.item_type,
       c.spend_area, c.gr_based, c.source,
       (clr.clear_ts IS NULL OR date_diff('day', c.start_ts, clr.clear_ts) > 126)::INT AS is_late,
       CASE WHEN c.start_ts < TIMESTAMPTZ '2018-07-01' THEN 'train' ELSE 'test' END AS split
FROM cases c JOIN clr USING (case_id)
WHERE c.start_ts >= TIMESTAMPTZ '2018-01-01'
  AND c.start_ts <  TIMESTAMPTZ '2018-09-01'
""")
con.sql("""
SELECT split, COUNT(*) AS cases, ROUND(100.0 * AVG(is_late), 1) AS pct_late
FROM ml_cases GROUP BY 1 ORDER BY 1
""").df()

,split,cases,pct_late
0,test,41268,24.2
1,train,134739,23.2


### First model (includes Consignment orders)
The score looks good, but the next check shows it is inflated.

In [11]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = con.sql("SELECT * FROM ml_cases").df()
features = ["company", "doc_type", "item_category", "item_type",
            "spend_area", "gr_based", "source"]
df[features] = df[features].astype(str)          # cast everything to text (categories)

train = df[df.split == "train"]
test  = df[df.split == "test"]
X_train, y_train = train[features], train.is_late
X_test,  y_test  = test[features],  test.is_late

# turn each category into 0/1 columns (one-hot)
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), features)])

models = {
    "Baseline (same probability for all)": DummyClassifier(strategy="prior"),
    "Logistic regression":       LogisticRegression(max_iter=1000),
}
for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:30s} ROC-AUC {roc_auc_score(y_test, p):.3f}   "
          f"PR-AUC {average_precision_score(y_test, p):.3f}")
print("Late rate in test set:", round(y_test.mean(), 3))

Baseline (same probability for all)    ROC-AUC 0.500   PR-AUC 0.242
Logistic regression          ROC-AUC 0.748   PR-AUC 0.580
Late rate in test set: 0.242


## 4. Check: what is the model learning?
Consignment orders are never invoiced (100% never cleared), so they are trivially "late".

In [12]:
con.sql("""
WITH clr AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
  FROM events_ord GROUP BY 1)
SELECT m.item_category,
       COUNT(*)                                         AS n,
       ROUND(100.0 * AVG(m.is_late), 1)                 AS pct_late,
       ROUND(100.0 * AVG((clr.clear_ts IS NULL)::INT), 1) AS pct_never_cleared
FROM ml_cases m JOIN clr ON clr.case_id = m.case_id
GROUP BY 1 ORDER BY n DESC
""").df()

,item_category,n,pct_late,pct_never_cleared
0,"3-way match, invoice before GR",154833,17.9,6.7
1,"3-way match, invoice after GR",10531,28.0,22.5
2,Consignment,10222,100.0,100.0
3,2-way match,421,76.2,64.4


## 5. Final cohort: standard 3-way-match orders only

In [14]:
con.execute("""
CREATE OR REPLACE TABLE ml_cases AS
WITH clr AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
  FROM events_ord GROUP BY 1)
SELECT c.case_id, c.start_ts,
       c.company, c.doc_type, c.item_category, c.item_type,
       c.spend_area, c.gr_based, c.source,
       (clr.clear_ts IS NULL OR date_diff('day', c.start_ts, clr.clear_ts) > 126)::INT AS is_late,
       CASE WHEN c.start_ts < TIMESTAMPTZ '2018-07-01' THEN 'train' ELSE 'test' END AS split
FROM cases c JOIN clr USING (case_id)
WHERE c.start_ts >= TIMESTAMPTZ '2018-01-01'
  AND c.start_ts <  TIMESTAMPTZ '2018-09-01'
  AND c.item_category LIKE '3-way match%'
""")
con.sql("""
SELECT split, COUNT(*) AS cases, ROUND(100.0 * AVG(is_late), 1) AS pct_late
FROM ml_cases GROUP BY 1 ORDER BY 1
""").df()

,split,cases,pct_late
0,test,38899,19.6
1,train,126465,18.2


### Model without vendor and order value

In [15]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = con.sql("SELECT * FROM ml_cases").df()
features = ["company", "doc_type", "item_category", "item_type",
            "spend_area", "gr_based", "source"]
df[features] = df[features].astype(str)          # cast everything to text (categories)

train = df[df.split == "train"]
test  = df[df.split == "test"]
X_train, y_train = train[features], train.is_late
X_test,  y_test  = test[features],  test.is_late

# turn each category into 0/1 columns (one-hot)
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), features)])

models = {
    "Baseline (same probability for all)": DummyClassifier(strategy="prior"),
    "Logistic regression":       LogisticRegression(max_iter=1000),
}
for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:30s} ROC-AUC {roc_auc_score(y_test, p):.3f}   "
          f"PR-AUC {average_precision_score(y_test, p):.3f}")
print("Late rate in test set:", round(y_test.mean(), 3))

Baseline (same probability for all)    ROC-AUC 0.500   PR-AUC 0.196
Logistic regression          ROC-AUC 0.671   PR-AUC 0.361
Late rate in test set: 0.196


## 6. Add vendor and order value (both known at order creation)

In [16]:
con.execute("""
CREATE OR REPLACE TABLE ml_cases2 AS
WITH extra AS (
  SELECT "case:concept:name" AS case_id,
         ANY_VALUE("case:Vendor") AS vendor,
         first("Cumulative net worth (EUR)" ORDER BY "time:timestamp", rn) AS net_worth
  FROM events_ord GROUP BY 1)
SELECT m.*, e.vendor, e.net_worth
FROM ml_cases m JOIN extra e USING (case_id)
""")
con.sql("""
SELECT COUNT(*) AS cases,
       COUNT(DISTINCT vendor) AS vendors,
       ROUND(MEDIAN(net_worth), 1) AS median_worth,
       SUM((net_worth IS NULL)::INT) AS missing_worth
FROM ml_cases2
""").df()

,cases,vendors,median_worth,missing_worth
0,165364,1543,573.0,0.0


In [17]:
import numpy as np
from sklearn.impute import SimpleImputer

df = con.sql("SELECT * FROM ml_cases2").df()
cat_features = ["company", "doc_type", "item_category", "item_type",
                "spend_area", "gr_based", "source", "vendor"]
df[cat_features] = df[cat_features].astype(str)
df["log_worth"] = np.log1p(df["net_worth"].clip(lower=0))   # order values are very skewed: take the log

num_features = ["log_worth"]
train = df[df.split == "train"]
test  = df[df.split == "test"]
X_train, y_train = train[cat_features + num_features], train.is_late
X_test,  y_test  = test[cat_features + num_features],  test.is_late

prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", SimpleImputer(strategy="median"), num_features),
])
models = {
    "Baseline (same probability for all)": DummyClassifier(strategy="prior"),
    "Logistic regression":       LogisticRegression(max_iter=2000),
}
for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:30s} ROC-AUC {roc_auc_score(y_test, p):.3f}   "
          f"PR-AUC {average_precision_score(y_test, p):.3f}")
print("Late rate in test set:", round(y_test.mean(), 3))

Baseline (same probability for all)    ROC-AUC 0.500   PR-AUC 0.196
Logistic regression          ROC-AUC 0.815   PR-AUC 0.626
Late rate in test set: 0.196


### Gradient boosting comparison
No improvement over logistic regression, so the simple model is kept.

In [18]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

train = df[df.split == "train"]
top_vendors = train.vendor.value_counts().head(200).index        # computed on the training set only
df["vendor_g"] = df.vendor.where(df.vendor.isin(top_vendors), "other")

cat2 = [c for c in cat_features if c != "vendor"] + ["vendor_g"]
train = df[df.split == "train"]
test  = df[df.split == "test"]
X_train, y_train = train[cat2 + num_features], train.is_late
X_test,  y_test  = test[cat2 + num_features],  test.is_late

prep_gb = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat2),
    ("num", "passthrough", num_features),
])
gb = HistGradientBoostingClassifier(
    categorical_features=list(range(len(cat2))),
    max_iter=300, learning_rate=0.08,
    early_stopping=True, random_state=0)

pipe = Pipeline([("prep", prep_gb), ("clf", gb)])
pipe.fit(X_train, y_train)
p = pipe.predict_proba(X_test)[:, 1]
print(f"Gradient boosting   ROC-AUC {roc_auc_score(y_test, p):.3f}   "
      f"PR-AUC {average_precision_score(y_test, p):.3f}")

Gradient boosting   ROC-AUC 0.808   PR-AUC 0.619


## 7. Which features matter? (ablation: remove one feature at a time)

In [20]:
def fit_eval(cat_cols, num_cols):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols),
    ])
    pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))])
    pipe.fit(train[cat_cols + num_cols], train.is_late)
    p = pipe.predict_proba(test[cat_cols + num_cols])[:, 1]
    return average_precision_score(test.is_late, p)

all_cat = cat_features            # includes "vendor" (not vendor_g)
full = fit_eval(all_cat, num_features)
print(f"Full model: PR-AUC {full:.3f}\n")

rows = []
for col in all_cat:
    score = fit_eval([c for c in all_cat if c != col], num_features)
    rows.append((col, score, full - score))
score = fit_eval(all_cat, [])
rows.append(("log_worth (order value)", score, full - score))

res = pd.DataFrame(rows, columns=["feature removed", "PR-AUC without it", "loss"])
res.sort_values("loss", ascending=False).round(3)

Full model: PR-AUC 0.626



,feature removed,PR-AUC without it,loss
7,vendor,0.384,0.242
8,log_worth (order value),0.613,0.013
4,spend_area,0.624,0.002
3,item_type,0.624,0.002
0,company,0.625,0.001
2,item_category,0.625,0.001
1,doc_type,0.625,0.000
5,gr_based,0.625,0.000
6,source,0.625,0.000


## 8. What does the model detect: slow payments or missing invoices?

In [21]:
con.sql("""
WITH clr AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
  FROM events_ord GROUP BY 1)
SELECT COUNT(*)                                             AS late_cases,
       SUM((clr.clear_ts IS NULL)::INT)                     AS never_paid,
       ROUND(100.0 * AVG((clr.clear_ts IS NULL)::INT), 1)   AS pct_never_paid_among_late
FROM ml_cases m JOIN clr ON clr.case_id = m.case_id
WHERE m.is_late = 1
""").df()

,late_cases,never_paid,pct_never_paid_among_late
0,30668,12750.0,41.6


In [22]:
con.sql("""
WITH v AS (
  SELECT vendor, COUNT(*) AS n, AVG(is_late) AS late_rate
  FROM ml_cases2 GROUP BY 1 HAVING COUNT(*) >= 200)
SELECT CASE WHEN late_rate < 0.10 THEN '1) under 10% late'
            WHEN late_rate < 0.30 THEN '2) 10-30% late'
            WHEN late_rate < 0.60 THEN '3) 30-60% late'
            ELSE                       '4) over 60% late' END AS bucket,
       COUNT(*) AS vendors, SUM(n) AS cases
FROM v GROUP BY 1 ORDER BY 1
""").df()

,bucket,vendors,cases
0,1) under 10% late,82,68477.0
1,2) 10-30% late,30,41124.0
2,3) 30-60% late,10,15116.0
3,4) over 60% late,9,7170.0


In [23]:
# Refit the final logistic regression (all features)
prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ("num", SimpleImputer(strategy="median"), num_features),
])
lr = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))])
lr.fit(train[cat_features + num_features], train.is_late)

test = test.copy()
test["p"] = lr.predict_proba(test[cat_features + num_features])[:, 1]

# Clearing date for each case
clr = con.sql("""
SELECT "case:concept:name" AS case_id,
       MIN("time:timestamp") FILTER (WHERE "concept:name" = 'Clear Invoice') AS clear_ts
FROM events_ord GROUP BY 1
""").df()
test = test.merge(clr, on="case_id", how="left")

days = (test.clear_ts - test.start_ts).dt.days
test["group"] = np.where(test.clear_ts.isna(), "never paid",
                  np.where(days > 126, "paid late", "paid on time"))

test.groupby("group").agg(cases=("p", "size"),
                          mean_predicted_risk=("p", "mean")).round(3)

,cases,mean_predicted_risk
group,,
never paid,3869,0.500
paid late,3723,0.378
paid on time,31307,0.117
